# Research state — recursive acquisition of executable knowledge
**2026-11-25 | revision 4 | read this before changing architecture.** This is the cumulative design/decision record, not a claim of a novel algorithm or a successful ARC system. Runnable experiment: [experiment_04.ipynb](./experiment_04.ipynb). Original neural source: [recursive_world.py](./recursive_world.py). Original GPU runner: [colab.ipynb](./colab.ipynb). Preserve the original notebooks and raw outputs; do not overwrite old runs. The separate downloadable cumulative ledger v4 contains the historical Kaggle outputs and the optional neural integration code; this GitHub state notebook records their findings but does not embed their entire raw output archive. 
## 1. Research question
Can an agent acquire executable knowledge from evidence, use acquired knowledge as a tool to acquire further knowledge, and revise previously acquired concepts and their uses when subsequent evidence contradicts them? Specifically: does learning a distribution of reusable **transformations of programs**, then learning when to reuse/revise these transformations, reduce sample or search complexity relative to strong nonrecursive alternatives?
This is not an attempt to implement Sanskrit grammar, replace neural networks, demonstrate AGI, or win a synthetic benchmark through a favorable generator. Deterministic execution is conditional on having a valid hypothesis; inference about which hypothesis applies is uncertain. A label is useful if it denotes an executable operation, predicate, or other semantically testable object, not just a slot in a tensor.
Inspiration: Pāṇini's Aṣṭādhyāyī and the later grammatical tradition use definitions, abbreviations, rule application conditions, exceptions, and interacting rules. Bronkhorst discusses the dependence between 1.3.3 and 1.1.71 and the problem of arranging the exposition linearly. The idea that later knowledge can change earlier interpretations motivates our design; the historical texts do not establish a modern learning algorithm. Other reading leads: Patañjali's Mahābhāṣya (general rule/exception and jñāpaka); Bhartṛhari's Vākyapadīya 1.82–85 (repeated cognition; consult edition/commentary before making textual claims). No strong historical equivalence is asserted.

## Architecture at a glance
![Architecture comparison](architecture.svg)
[Open SVG](architecture.svg). The three columns distinguish discarded E1, tested E2–E4 components, and the proposed full recursive acquisition system. [Complete notebook index](README.md) · [Cumulative ledger with original outputs](experiment_ledger_v4.ipynb).

## 2. Target architecture (proposed, not implemented as a whole)
```
observations + actions --> pretrained perception / neural predictor ----> default prediction
             |                              |                                   |
             v                              v                                   |
          object/state hypotheses --> typed program proposals                   |
             |                           ^                                      |
             v                           |                                      |
       versioned theory <------ transformation library + search policy          |
             |                           ^                                      |
             v                           |                                      |
       execute / compare / abstain ------+-------------------------------> prediction
             |
       observe actual result (after prediction only)
             |
       retain contradictory evidence, update version spaces, revise dependencies
             |
       learn which revisions worked -> reusable higher-order operators / policy
```
Interface contract:
- Observation record: immutable state/action/next-state, history, episode identity, provenance. No future outcomes or hidden game code at prediction.
- Concept: typed predicate or value-producing program, support, counterexamples, dependencies and version. Example: an obstacle predicate applicable to multiple object types, not one color token.
- Operation: pure executable program mapping typed state and parameters to effects; applicability guard is separate from effect. Identity is always a candidate, not assumed globally correct.
- Rule: guard + operation + scope, with a version space of equivalent/ambiguous alternatives. Distinguish observed consistency from truth outside the evidence. Preserve counterexamples.
- Transformation: typed function on programs or theories (generalize guard, specialize on context, compose, replace subexpression, introduce exception, factor repeated subexpression). Example: specialize move by not-obstacle; generalize obstacle from color to collision relation; reuse that specialization operation on a new action rule.
- Library: versioned DAG of definitions and their dependencies; store stable identifiers and immutable definitions. Revisions create new versions rather than silently modifying historical meaning. Revalidate dependent candidates on stored evidence; do not retroactively rewrite prequential scores.
- Acquisition policy: choose what programs or transformations to propose under a fixed search/compute budget. Learn from verified successful/failed transformations on completed tasks; use existing DreamCoder-style enumeration/library learning as a strong control. Novelty would require benefit beyond ordinary learned search priors, abstraction compression, retrieval and recurrent latent-context inference.
- Execution: optional override of strong neural prediction only if evidence-calibrated rule proposals agree; otherwise abstain. Do not force all perception through a symbolic bottleneck.
- Complexity control: deduplicate by observed behavior, keep uncertainty over extrapolation, penalize library size and search cost; birth only after predictive or compression evidence. No arbitrary slot splitting. Context creation is not concept discovery.
Proposed higher-order loop: L0 observations; L1 executable operations; L2 guarded rules; L3 reusable program transformations; L4 procedures that select/revise transformations. Each layer must be usable by the next as an input, not just repeatedly invoke the same learned neural block. No claim of unbounded recursion. Currently only finite executable rewrite candidates, learned joint priors, hard context selection, and fixed residual guards exist. General dependency revision, new operator discovery, learned meta-operators and object perception are open.

## 3. History: measured results and decisions
### E1: recurrent neural dictionary (archived)
Three frames + action encoding -> convolutional encoder -> 3 shared recurrent refinements, each reading current canvas and previous label field -> applicability logits over K=12 label slots -> learned soft mixture over 13 local copy offsets (including identity) + 16 literal colors -> execute -> next pass. Start with four active slots for grow/small/stale. Split highest-responsibility slot on errors every four updates; posterior relabel old data using current executor vs stale control. WARC frozen teacher optional; direct adaptation control shares input and mixture gate initialized toward teacher. Balanced changed/unchanged NLL, AdamW, 20 updates per transition, replay newest 10 + 20 older with rotations; BF16 only on natively supporting devices, T4 FP32. Profiling, weight drift, vocabulary, prediction metrics. Source pinned at 2529f513d3eb68b26295d9ddf69d4346b2a0e4c4; Kaggle locally patched the shadowed prior variable and native BF16 detection, so that exact pinned source is not a runnable replacement for the local patched notebook.
| ls20, 12 exposures | changed accuracy | false-edit rate | update seconds |
|---|---:|---:|---:|
| direct |58.57%|12.71%|0.370|
| copy |54.02%|7.51%|0.505|
| tiny WARC scratch |38.46%|9.15%|0.776|
| small dictionary |29.37%|6.34%|0.529|
| grow |29.20%|6.34%|0.536|
| stale |25.00%|10.12%|0.542|
| ls20, 24 exposures | changed accuracy | false-edit rate | change F1 |
|---|---:|---:|---:|
| WARC frozen |78.56%|0.12%|0.8185|
| WARC + direct |86.54%|0.76%|0.7242|
| WARC + grow |79.37%|0.14%|0.8261|
| direct scratch |69.65%|6.86%|0.4797|
| copy |67.31%|4.23%|0.5022|
| grow scratch |32.29%|5.09%|0.1902|
One recording, one seed, changed-cell accuracy averaged across transitions with changes, not official MechEval. No learned arm obtained exact frames; full WARC TTT missing. Tiny WARC is not compute matched. Hard grow/small/stale execution on synthetic unseen colors achieved zero changed accuracy. Labels grew from 6 to 8 (epochs 8 to 24), highest primitive confidence only 27%, duplicate primitives. Convolution dominated one profiled update; relative drift exploded at near-zero denominator. Tensor variable shadowing and emulated BF16 caused runtime bugs. Mechanism failed: expressive bottleneck, nonidentifiable soft labels, arbitrary births and gradient-based relabeling are not genuine higher-order acquisition. These are plausible failure explanations, not individually proven causal effects. **Discard executor; keep strong neural controls, data parsing, causal harness, and profiling only after correctness.**
### E2: abstaining executable residual
Keep WARC frozen and direct adapter; cache predictions once. Rule grammar: literal or relative copy effects guarded by action, source color, click relation, neural predicted color and optionally history/neighbors. Intersect consistent candidate effects across all observed matching cells; require at least two supporting error transitions. Contradictions revoke broad rules; surviving specializations may apply. If surviving programs disagree or guards conflict, abstain. No GPU required for rule tests. CPU discovery, contradiction/revocation, specialization, unseen-grid execution and prefix invariance passed in local notebook. Real ARC benefit not measured. Fixed grammar, no arbitrary composition or metarule discovery. Forecast on original pilot +1 pp vs direct, gate +2 pp without increased false edits or reduced F1/exact on ls20 and ka59; no large sweep without passing. Compare frozen baseline and simple agreement (edit only when both neural predictions edit).
### E3: finite higher-order learning
Four-cell, three-color inputs (81 exhaustive truth-table rows). Source programs drawn from 8 primitives; training source set left/cycle/swap01/reverse; test right/uncycle/swap12/swap_halves. Transformations: 8 input precompositions, 8 output postcompositions, 3 color guards, identity (20). Candidate bank 20 singles + 400 ordered pairs. Learn fractional counts of transformations consistent with fully observed source and completed target, using 48 meta-training tasks. Factorized control preserves row/column marginals and total mass of joint pair counts, discards correlation. One target example before prediction, unused grids scored. Replication: one-step uniform 81.7%, learned rewrite 91.1%; factorized composition 85.7%, joint 98.9%; +13.18 pp. Under swapped pairs, joint 54.3%, factorized 78.9% (−24.63 pp). This is prior learning over a pre-enumerated bank deliberately containing correlated tasks, not discovering a new DSL or proving search-time advantage. Candidate enumeration work is the same. Demonstrates a limited transferable statistical structure and vulnerability to shift.

## 4. Current runnable algorithm: E4 context reuse
See [experiment_04.ipynb](./experiment_04.ipynb) for full code. Same finite E3 program bank. M=20, D=420, uniform U=1/D. All learned models initialize from identical 48 completed meta-training tasks (seed 101); uniform receives no prior. Per test task: infer fully observed source program; receive one target example; compute Boolean consistent mask A; choose the highest-prior consistent program (shared random tie-break); evaluate exact output on 73 held-out inputs (7 other inputs withheld too); **only then** receive full target truth table, compute full consistency mask E, and update with fractional credit E/sum(E). Full feedback is generous and not ARC-like. Do not use withheld target outputs to predict or select hyperparameters.
Controls: fixed uniform; fixed joint; fixed factorized; forgetting c <- 0.9 c + credit; change-point filter (hazard 0.05); hard context mixture joint and factorized. All use normalization of counts plus U. Factorized pair matrix is outer product of row and column sums divided by total pair mass.
Change-point control maintains run-length weights w_r and counts c_r; predictive prior (1-h) sum_r w_r norm(c_r+U) + hU. Evidence likelihood of each component is dot(prior,E); after reveal multiply weights by likelihood, prepend reset component, normalize, update counts. Fractional-credit approximation, not exact conjugate inference over ambiguous programs.
Context model stores count vector per regime. Prior weights: current regime 0.90, each of k−1 other regimes and a new regime 0.10/k. New regime uses U. Predict by mixture, then choose MAP component by weight times full evidence likelihood; update its counts (or create new one); retain old count vectors. No supplied context identifiers. This is hand-specified hard assignment and fixed stickiness, not learned context revision or dependency reinterpretation. 
Protocol: 60 tasks/stream, stable A, A-B-A (20 each), random candidate stream. 48 streams per seed; pilot 20261129 and independent 20261230; intermediate 20261130 overlaps 47 streams and is excluded from replication claims. Recovery = first offset with three consecutive across-stream mean exact scores >=90%. Forecast: stable >=96%, first shift <=10 tasks, return <=2, no random advantage.
| seed | method | A | B | A return | first shift / return offsets |
|---|---|---:|---:|---:|---|
|20261129|fixed joint|96.3|56.4|96.0|—|
||forget|96.5|84.7|93.0|14/4|
||change point|96.2|87.4|93.3|9/4|
||context joint|96.2|85.6|95.3|13/2|
|20261230|fixed joint|95.7|58.8|96.6|—|
||forget|95.4|85.9|93.7|10/4|
||change point|95.6|88.9|93.5|10/4|
||context joint|95.6|86.1|96.4|12/1|
All values except offsets are percent exact on unseen inputs. Context vs change-point paired mean in ABA +0.08 pp (95% stream-bootstrap interval −0.49 to +0.67), +0.04 pp (−0.53 to +0.57). Random −2.19 pp (−3.07 to −1.25), −2.30 pp (−3.32 to −1.32); context creates about 3 extra regimes under random evidence. Stable phase average about 96.0/96.2. Context factor never recovers by threshold. Unit tests: normalization, contradiction, new context, old context return. CPU ~19s/seed with single-thread BLAS after initial timeout. No overall win; reactivation improves but first-shift gate fails twice. Do not scale. Interpretation: hard assignment and stickiness delay adaptation; spurious contexts under random evidence indicate need for uncertainty/complexity control. This explanation requires further tests.

## 5. Next experiments and stopping rules
E5a: compare soft latent regime responsibility and evidence-calibrated reactivation with hard MAP, forgetting, BOCPD and a finite mixture/HMM baseline. Use identical banks/feedback and fresh disjoint seed ranges. Include uniform and fixed joint. Declare hazard/stickiness and any context penalty before evaluation; tune only on separate development streams. Report stable, first shift, return, random, full-stream paired regret, birth count and compute. Gate: stable >=96%; first-shift <=10; return <=2; paired overall advantage over change point with stream-level uncertainty excluding zero, no meaningful random degradation (define margin before test). Compare to a strong no-discovery recurrent-regime model. If no advantage, do not endlessly tune context selection; treat this as known continual inference, not research novelty.
E5b: actual library induction. Use a typed enumerator over primitive ASTs, not a pre-listed macro table. Train on shorter expressions; withhold entire operator compositions and longer trees; compare flat enumerator, frequency prior, factorized learned search, DreamCoder-like compression, retrieval, and learned rewrite proposal. Admit a macro only if it compresses prior solved tasks and improves prediction/search on independent tasks, accounting for library cost. Record exact unseen examples, number of candidate executions, search wall time, exposures, memory. Require fixed-budget benefit on two independently designed domains and adversarial nonreusable tasks. Distinguish semantic equivalence on a finite domain from general equivalence.
E6: revision with dependencies and incomplete evidence. Counterexample should invalidate a broad rule, retain a consistent special case, recheck dependent rules, and improve acquisition of later tasks. Test delayed contradictions, ambiguous evidence, random/noisy task feedback, out-of-library tasks and order dependence. Ablate retrospective revalidation separately from context reuse and macro induction. Store old version and causal predictions. 
E7: real ARC replay only after these gates. Use pretrained WARC/adapter without retraining each CPU ablation. Match streams, held-out fold, augmentation, updates, parameter budget and report wall time. Compare strong released WARC TTT, frozen, direct adapter, keep, retrieval, agreement, optional rule correction. Real changed-cell accuracy, false edit rate, change F1, exact, per-session/per-game aggregation, paired intervals. Official MechEval and proxy are distinct: official labels/scorer unavailable in prior inspection; don't compare numbers. Need benchmark official data identity, labels/scorer, multiple held-out games/seeds, compute and exposure matching before SOTA claim. Do not scale T4 experiments until simple gates pass.
Experimental discipline: write predictions and criteria first; compile and run CPU tests, use known simple deterministic tasks to test semantics, inspect failures, hold out new inputs/tasks, do not change several mechanisms at once, don't train to fit the evaluation set. Optimize only measured bottlenecks after evidence of benefit. Keep raw logs, code commit, seeds, config, unit test results, negative results, metric definitions, dataset version, and exact decisions. If a hypothesis is falsified, say so. No guaranteed win.

## 6. References and direct relevance
- Johannes Bronkhorst, *From Pāṇini to Patañjali: The Search for Linearity* (2004), [text](https://www.academia.edu/3279193/From_P%C4%81%E1%B9%87ini_to_Pata%C3%B1jali_the_search_for_linearity). Historical inspiration for mutually dependent definitions and interpretation order, not empirical evidence.
- Ellis et al., [DreamCoder: Growing generalizable, interpretable knowledge with wake-sleep Bayesian program learning](https://arxiv.org/abs/2006.08381); [PLDI version](https://doi.org/10.1145/3453483.3454080). Learns abstractions and neural program search; essential comparison. Existing library learning already realizes substantial parts of the intuition; novelty cannot be simply learning compositions.
- Wong et al., [Leveraging Language to Learn Program Abstractions and Search Heuristics](https://proceedings.mlr.press/v139/wong21a.html). Relevant to separate learned search from learned library.
- Adams & MacKay, [Bayesian Online Changepoint Detection](https://arxiv.org/abs/0710.3742). E4 comparator inspired by run-length filtering, not verbatim exact Bayesian implementation.
- Fox et al., sticky HDP-HMM, [paper search starting point](https://arxiv.org/abs/0905.2592). Recurrent latent regimes and self-transition bias are existing ideas; compare before claiming a new context learner. 
- [How Many Exposures Does a World Model Need? Learning Unseen Game Mechanics at Test Time](https://arxiv.org/abs/2611.12324); [WARC upstream](https://github.com/third-intelligence/WARC), pinned f0081ef00e867f63a08b1e4e7350d2baa0440267. 35M vision transformer, held-out game folds, 20 online updates per transition, replay and rotations; official MechEval 69 mechanics across 20 games. Original source inspected: agent/warc_model/model.py, final_ttt.py, varc2_optim.py, agent/VARC/agent_base/src/augment.py, hist_window/src/history.py, docs/reproducibility.md. Published MechEval numbers reference only (All%): frozen 18.4, scratch 35.0, TTT 69.1, episodic 32.9, GRU 24.0, DreamerV3 25.9. Not comparable to changed-cell replay. 
- [ARC-AGI-3 benchmark technical report](https://arcprize.org/media/ARC_AGI_3_Technical_Report.pdf). Interactive exploration/planning benchmark; next-frame replay is a narrower diagnostic.
- [fast.ai practical deep learning](https://course.fast.ai/) and [DawnBench](https://dawn.cs.stanford.edu/benchmark/): methodology inspiration for simple baselines, small-scale debugging, predicting outcomes, measurement before optimization. Not evidence for our proposed algorithm.
Further reading queue (not verified or integrated into current code): predicate invention/ILP, minimum description length, continual-learning mixture models, executable world models for ARC-AGI-3. Check prior art before claims. 
**State at end of revision 4:** E3 controlled transfer succeeds; E4 rapid return succeeds but overall shift adaptation fails; real-data rule residual unmeasured; full recursive knowledge acquisition and novelty unestablished. No GPU run authorized by these results.

## E4 stress audit and superseding decision
[Executable audit and captured results](experiment_04_stress.ipynb). 24 independent streams per regime, seeds 20270101–20270124, irregular A9/B13/A5/B9 and random tasks; one example before scoring, 1/4/81 total feedback examples afterward. Context minus change point: −0.36 pp [−1.88,1.11], −0.18 pp [−1.27,0.87], +0.03 pp [−1.05,1.10]. Random four-feedback gap −1.70 pp [−3.42,−0.01]. No reliable overall advantage. **Supersedes the earlier plan to keep tuning context allocation: prioritize typed AST enumeration and genuine library induction.** A soft context variant may later be a control rather than the main research contribution. Compare DreamCoder and Stitch-style compression ([paper](https://arxiv.org/abs/2211.16605), [code](https://github.com/mlb2251/stitch)). Fixed candidate ranking is not program discovery. The public benchmark plan and claim restrictions are in [README.md](README.md). No SOTA claim, no new GPU run.